In [1]:
import langchain
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader,TextLoader,CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\nabee\AppData\Local\Temp\ipykernel_17648\216620657.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,TextLoader,CSVLoader
d:\Test\Python\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class DocumentLoader:

    LOADERS = {
        ".pdf": PyMuPDFLoader,
        ".txt": TextLoader,
        ".csv":CSVLoader
    }

    @classmethod
    def get_loader(cls,file):
        loader_class = cls.LOADERS.get(file.suffix.lower())

        if not loader_class:
            raise ValueError("Unsupported file type: ",file.suffix)

        return loader_class(str(file))

In [3]:

def process_all_files(path):
    all_docs = []

    file_path = Path(path)
    files = list(file_path.glob("**/*"))

    if not files:
        return []

    print(f"{len(files)} files found")

    for file in files:
        if not file.is_file():
            continue

        try:
            loader = DocumentLoader.get_loader(file)
            docs = loader.load()

            all_docs.extend(docs)

            print(f"Loaded: {file.name}")

        except ValueError:
            print(f"Skipping unsupported file: {file.name}")

        except Exception as e:
            print(f"Error loading {file.name}: {e}")

    return all_docs

In [4]:
all_docs = process_all_files("../data/")
all_docs

13 files found
Loaded: English_Teacher_AI_Prompts.pdf
Loaded: second.pdf
Loaded: machine_learning_intro.txt
Loaded: python_intro.txt
Skipping unsupported file: chroma.sqlite3
Skipping unsupported file: data_level0.bin
Skipping unsupported file: header.bin
Skipping unsupported file: length.bin
Skipping unsupported file: link_lists.bin


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-02T11:36:50+00:00', 'source': '..\\data\\pdf_files\\English_Teacher_AI_Prompts.pdf', 'file_path': '..\\data\\pdf_files\\English_Teacher_AI_Prompts.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-02T11:36:50+00:00', 'trapped': '', 'modDate': "D:20260602113650+00'00'", 'creationDate': "D:20260602113650+00'00'", 'page': 0}, page_content="English Teacher AI Prompt Collection\nProfessional prompts for lesson planning, grammar, reading, writing, speaking, vocabulary,\nliterature, assessment, and student support.\n1. Complete Lesson Plan\nAct as an experienced English teacher. Create a detailed 45-minute lesson plan on '[Topic]' for\nstudents of grade [X]. Include learning objectives, warm-up activity, explanation, guided practice,\nindependent practice, as

In [5]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                   chunk_overlap=chunk_overlap,
                                                   length_function=len,
                                                   separators=["\n\n","\n"," ",""])

    split_docs = text_splitter.split_documents(documents)
    return split_docs

In [37]:
chunks = split_documents(all_docs)
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-06-02T11:36:50+00:00', 'source': '..\\data\\pdf_files\\English_Teacher_AI_Prompts.pdf', 'file_path': '..\\data\\pdf_files\\English_Teacher_AI_Prompts.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-02T11:36:50+00:00', 'trapped': '', 'modDate': "D:20260602113650+00'00'", 'creationDate': "D:20260602113650+00'00'", 'page': 0}, page_content="English Teacher AI Prompt Collection\nProfessional prompts for lesson planning, grammar, reading, writing, speaking, vocabulary,\nliterature, assessment, and student support.\n1. Complete Lesson Plan\nAct as an experienced English teacher. Create a detailed 45-minute lesson plan on '[Topic]' for\nstudents of grade [X]. Include learning objectives, warm-up activity, explanation, guided practice,\nindependent practice, as

In [ ]:
import chromadb
from typing import List,Dict,Any,Tuple


class VectorStore:
    def __init__(self,collection_name: str = "pdf_document",persist_directory:str="../data/vector_store"):

        self.collectio_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collectio_name,
                metadata={"description":"PDF document embeddings for RAG",
                          "hnsw:space":"cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collectio_name.count()}")
            print(f"Existing documents in colleciton: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")

        def add_documents(self,documents:List[Any],)


IndentationError: expected an indented block after function definition on line 2 (945543552.py, line 4)